In [1]:
%pip install optbinning pandas xlrd duckdb



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from itertools import combinations
from optbinning import OptimalBinning


In [2]:
data = pd.read_excel("default of credit card clients.xls", header=1)
data = data.drop(columns=["ID"])
target = "default payment next month"
data.head()


,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,PAY_5,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
def target_value_counts(df, target_col, normalize=True, pct=True):
    counts = df[target_col].value_counts(normalize=normalize)
    return counts * 100 if pct else counts

def compute_iv_ranking(df, target, solver="cp"):
    results = []
    for col in df.columns:
        if col == target:
            continue
        try:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype, solver=solver)
            optb.fit(df[col].values, df[target].values)
            iv = optb.binning_table.build().IV.iloc[-1] 
            results.append({"variable": col, "iv": iv * 100})
        except Exception as exc:
            print(f"Skipping {col}: {exc}")
    return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

def create_binned_df(df, target, variables):
    binned_df = pd.DataFrame(index=df.index)
    for col in variables:
        dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
        optb = OptimalBinning(name=col, dtype=dtype)
        optb.fit(df[col].values, df[target].values)
        binned_df[col] = optb.transform(df[col], metric="bins")
    binned_df[target] = df[target].values
    return binned_df


In [4]:
def one_way_summary(binned_df, target, base_rate, pct=True):
    results = []
    for col in binned_df.columns:
        if col == target:
            continue
        summary = (
            binned_df
            .groupby(col)
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = col + "=" + summary[col].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def two_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2 in combinations(cols, 2):
        summary = (
            binned_df
            .groupby([c1, c2])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = c1 + "=" + summary[c1].astype(str) + sep + c2 + "=" + summary[c2].astype(str)
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def three_way_summary(binned_df, target, base_rate, pct=True, sep=" & "):
    results = []
    cols = [c for c in binned_df.columns if c != target]
    for c1, c2, c3 in combinations(cols, 3):
        summary = (
            binned_df
            .groupby([c1, c2, c3])
            .agg(count=(target, "size"), events=(target, "sum"))
            .reset_index()
        )
        summary["rate"] = summary["events"] / summary["count"] * 100
        summary["lift"] = summary["rate"] / (base_rate * 100) if pct else summary["rate"] / base_rate
        summary["rule"] = (
            c1 + "=" + summary[c1].astype(str) + sep +
            c2 + "=" + summary[c2].astype(str) + sep +
            c3 + "=" + summary[c3].astype(str)
        )
        results.append(summary[["rule", "count", "rate", "lift"]])
    return pd.concat(results).sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)

def shortlist_rules(df, min_sample_size=1000, min_lift=2.0, base_rate=None):
    if base_rate is None:
        raise ValueError("base_rate is required")
    threshold = base_rate * 100
    return df.assign(
        shortlist=(
            (df["count"] >= min_sample_size) &
            (df["lift"] >= min_lift) &
            (df["rate"] > threshold)
        ).astype(int)
    )


In [5]:
data.shape

(30000, 24)

In [6]:
iv_ranking = compute_iv_ranking(data, target)
top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(data, target, top_vars)
base_rate = data[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=...",1002,77.245509,3.492112,1
1,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf) & PAY_6=...",1072,77.238806,3.491809,1
2,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & PAY_6=...",1091,77.176902,3.489010,1
3,"PAY_0=[1.50, inf) & PAY_3=[1.50, inf) & PAY_6=...",1024,76.855469,3.474479,1
4,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf)",1249,76.621297,3.463892,1
5,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1053,76.448243,3.456069,1
6,"PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_5=...",1181,76.291279,3.448973,1
7,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1000,76.200000,3.444846,1
8,"PAY_0=[1.50, inf) & PAY_5=[1.00, inf)",1309,76.088617,3.439811,1
9,"PAY_0=[1.50, inf) & PAY_6=[1.00, inf) & BILL_A...",1011,75.964392,3.434195,1


In [7]:
binned_df.shape

(30000, 24)

In [8]:
def parse_rule(rule_str):
    parts = [p.strip() for p in rule_str.split("&")]
    conditions = []
    for part in parts:
        column, interval = part.split("=", 1)
        column = column.strip()
        interval = interval.strip()
        include_lower = interval.startswith("[")
        include_upper = interval.endswith("]")
        lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
        def to_bound(value):
            value = value.lower()
            if value in {"inf", "infty", "+inf", "+infty"}:
                return float("inf")
            if value in {"-inf", "-infty"}:
                return float("-inf")
            return float(value)
        conditions.append({
            "column": column,
            "lower": to_bound(lower_str),
            "upper": to_bound(upper_str),
            "include_lower": include_lower,
            "include_upper": include_upper,
        })
    return conditions

def build_mask(df, conditions):
    mask = pd.Series(True, index=df.index)
    for cond in conditions:
        column = cond["column"]
        column_values = pd.to_numeric(df[column], errors="coerce")
        current = pd.Series(True, index=df.index)
        if cond["include_lower"]:
            current &= column_values >= cond["lower"]
        else:
            current &= column_values > cond["lower"]
        if cond["include_upper"]:
            current &= column_values <= cond["upper"]
        else:
            current &= column_values < cond["upper"]
        mask &= current.fillna(False)
    return mask

first_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(first_rule)
mask = build_mask(data, conditions)
df_filtered = data[~mask]
df_filtered[target].value_counts(normalize=True) * 100


default payment next month
0    79.784813
1    20.215187
Name: proportion, dtype: float64

In [9]:
df_filtered.shape

(28998, 24)

In [11]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[1.50, inf) & BILL_AMT4=[5756.00, inf) &...",1040,69.230769,3.424691,1
1,"PAY_0=[1.50, inf) & PAY_3=[-0.50, 1.50)",1125,67.911111,3.359410,1
2,"PAY_0=[1.50, inf) & BILL_AMT6=[8814.00, inf) &...",1657,67.833434,3.355568,1
3,"PAY_0=[1.50, inf) & PAY_3=[-0.50, 1.50) & BILL...",1004,67.828685,3.355333,1
4,"PAY_0=[1.50, inf) & BILL_AMT5=[8499.50, inf) &...",1720,67.790698,3.353454,1
5,"PAY_0=[1.50, inf) & PAY_3=[-0.50, 1.50) & BILL...",1058,67.769376,3.352399,1
6,"PAY_0=[1.50, inf) & PAY_2=[1.50, inf) & BILL_A...",1051,67.745005,3.351194,1
7,"PAY_0=[1.50, inf) & BILL_AMT5=[8499.50, inf) &...",1652,67.675545,3.347758,1
8,"PAY_0=[1.50, inf) & BILL_AMT5=[8499.50, inf)",1737,67.587795,3.343417,1
9,"PAY_0=[1.50, inf) & BILL_AMT4=[5756.00, inf)",1826,67.524644,3.340293,1


In [12]:
second_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(second_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    81.608126
1    18.391874
Name: proportion, dtype: float64

In [13]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[0.50, inf) & PAY_2=[1.50, inf) & PAY_3=...",1382,52.966715,2.879898,1
1,"PAY_0=[0.50, inf) & PAY_4=[0.50, inf)",1030,52.912621,2.876957,1
2,"PAY_0=[0.50, inf) & PAY_2=[1.50, inf) & SEX=(-...",1352,52.292899,2.843261,1
3,"PAY_0=[0.50, inf) & PAY_3=[1.50, inf)",1481,51.654288,2.808539,1
4,"PAY_2=[1.50, inf) & PAY_3=[1.50, inf)",1582,51.517067,2.801078,1
5,"PAY_2=[1.50, inf) & SEX=(-inf, 1.50)",1531,51.143044,2.780741,1
6,"PAY_0=[0.50, inf) & PAY_4=[-0.50, 0.50) & SEX=...",1145,51.091703,2.777950,1
7,"PAY_0=[0.50, inf) & SEX=(-inf, 1.50) & MARRIAG...",1038,51.059730,2.776211,1
8,"PAY_0=[0.50, inf) & PAY_5=[-0.50, 1.00) & SEX=...",1287,50.893551,2.767176,1
9,"PAY_0=[0.50, inf) & PAY_6=[-0.50, 1.00) & SEX=...",1273,50.274941,2.733541,1


In [14]:
third_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(third_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    83.406081
1    16.593919
Name: proportion, dtype: float64

In [15]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_0=[0.50, inf) & PAY_3=(-inf, 0.50) & SEX=(...",1533,42.857143,2.582702,1
1,"PAY_0=[0.50, inf) & SEX=(-inf, 1.50)",1575,42.730159,2.575049,1
2,"PAY_0=[0.50, inf) & PAY_2=(-inf, 1.50) & SEX=(...",1051,42.721218,2.574510,1
3,"PAY_0=[0.50, inf) & PAY_3=(-inf, 0.50) & PAY_5...",1545,42.200647,2.543139,1
4,"PAY_0=[0.50, inf) & PAY_4=(-inf, 0.50) & SEX=(...",1507,41.871267,2.523290,1
5,"PAY_0=[0.50, inf) & PAY_5=[-0.50, inf)",1601,41.723923,2.514410,1
6,"PAY_0=[0.50, inf) & PAY_6=(-inf, 1.00) & SEX=(...",1434,41.422594,2.496251,1
7,"PAY_2=[1.50, inf) & PAY_4=(-inf, 0.50) & PAY_A...",1231,40.942323,2.467309,1
8,"PAY_2=[1.50, inf) & PAY_5=[-0.50, inf)",1105,40.904977,2.465058,1
9,"PAY_2=[1.50, inf) & PAY_AMT1=(-inf, 124.00)",1302,40.783410,2.457732,1


In [16]:
forth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(forth_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    85.013776
1    14.986224
Name: proportion, dtype: float64

In [17]:
# iv_ranking = compute_iv_ranking(df_filtered, target)
# top_vars = iv_ranking.loc[iv_ranking["iv"] >= 0.1, "variable"].tolist()
binned_df = create_binned_df(df_filtered, target, top_vars)
base_rate = df_filtered[target].mean()
one_way = one_way_summary(binned_df, target, base_rate)
two_way = two_way_summary(binned_df, target, base_rate)
three_way = three_way_summary(binned_df, target, base_rate)
one_way = shortlist_rules(one_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
two_way = shortlist_rules(two_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
three_way = shortlist_rules(three_way, min_sample_size=1000, min_lift=2.0, base_rate=base_rate)
all_shortlisted = pd.concat([one_way.loc[one_way["shortlist"] == 1,:], two_way.loc[two_way["shortlist"] == 1,:], three_way.loc[three_way["shortlist"] == 1,:]], ignore_index=True)
all_shortlisted = all_shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
all_shortlisted.head(20)


,rule,count,rate,lift,shortlist
0,"PAY_2=[-0.50, inf) & PAY_AMT1=(-inf, 21.50)",1107,32.791328,2.188098,1
1,"PAY_2=[-0.50, inf) & PAY_4=(-inf, 0.50) & PAY_...",1036,32.432432,2.164150,1
2,"PAY_0=[0.50, inf) & PAY_3=(-inf, 0.50) & PAY_A...",1210,32.066116,2.139706,1
3,"PAY_0=[0.50, inf) & PAY_AMT1=(-inf, 21.50)",1211,32.039637,2.137939,1
4,"PAY_0=[0.50, inf) & PAY_AMT1=(-inf, 21.50) & S...",1211,32.039637,2.137939,1
5,"PAY_0=[0.50, inf) & PAY_4=(-inf, 0.50) & PAY_A...",1194,31.742044,2.118082,1
6,"PAY_4=[0.50, inf) & PAY_5=[-0.50, inf)",1081,31.637373,2.111097,1
7,"PAY_2=[-0.50, inf) & PAY_4=[0.50, inf)",1029,31.584062,2.107540,1
8,"PAY_2=[-0.50, inf) & PAY_3=[0.50, inf)",1075,31.441860,2.098051,1
9,"PAY_3=[0.50, inf) & PAY_5=[-0.50, inf)",1103,31.278332,2.087139,1


In [18]:
fifth_rule = all_shortlisted.loc[0, "rule"]
conditions = parse_rule(fifth_rule)
mask = build_mask(df_filtered, conditions)
df_filtered = df_filtered[~mask]
df_filtered[target].value_counts(normalize=True) * 100

default payment next month
0    85.837233
1    14.162767
Name: proportion, dtype: float64

In [19]:
import numpy as np

In [20]:
def convert_interval_to_query(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a pandas .query() compatible string.
    """
    query_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column
        if col_conditions:
            query_conditions.append(" & ".join(col_conditions))
            
    # Combine all column rules into the final query string
    return " & ".join(query_conditions)

# --- Test the converter ---
input_string = first_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=[1.00, inf)
Output: PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00


In [21]:
# --- Test the converter ---
input_string = second_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[1.50, inf) & BILL_AMT4=[5756.00, inf) & SEX=[1.50, inf)
Output: PAY_0 >= 1.50 & BILL_AMT4 >= 5756.00 & SEX >= 1.50


In [22]:
# --- Test the converter ---
input_string = third_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[0.50, inf) & PAY_2=[1.50, inf) & PAY_3=[1.50, inf)
Output: PAY_0 >= 0.50 & PAY_2 >= 1.50 & PAY_3 >= 1.50


In [23]:
# --- Test the converter ---
input_string = forth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_0=[0.50, inf) & PAY_3=(-inf, 0.50) & SEX=(-inf, 1.50)
Output: PAY_0 >= 0.50 & PAY_3 < 0.50 & SEX < 1.50


In [24]:
# --- Test the converter ---
input_string = fifth_rule
generated_query = convert_interval_to_query(input_string)

print("\n--- Auto-Generated Query String ---")
print(f"Input:  {input_string}")
print(f"Output: {generated_query}")


--- Auto-Generated Query String ---
Input:  PAY_2=[-0.50, inf) & PAY_AMT1=(-inf, 21.50)
Output: PAY_2 >= -0.50 & PAY_AMT1 < 21.50


In [25]:
data['default payment next month'].value_counts(normalize=True) * 100

default payment next month
0    77.88
1    22.12
Name: proportion, dtype: float64

In [23]:
data.query("PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00")[target].value_counts(normalize=True) * 100

default payment next month
1    77.245509
0    22.754491
Name: proportion, dtype: float64

In [24]:
subset_data = data[~data.index.isin(data.query("PAY_0 >= 1.50 & PAY_4 >= 0.50 & PAY_6 >= 1.00").index)]

In [25]:
subset_data.query("PAY_0 >= 1.50 & BILL_AMT4 >= 5756.00 & SEX >= 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    69.230769
0    30.769231
Name: proportion, dtype: float64

In [26]:
data.query("PAY_0 >= 1.50 & PAY_2 >= 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    71.341748
0    28.658252
Name: proportion, dtype: float64

In [27]:
data.query("PAY_0 >= 0.50 & PAY_3 >= -0.50 & PAY_3 < 1.50")[target].value_counts(normalize=True) * 100

default payment next month
1    53.981436
0    46.018564
Name: proportion, dtype: float64

In [34]:
subset_data = data[~data.index.isin(data.query("PAY_0 >= 0.50 & PAY_3 >= -0.50 & PAY_3 < 1.50").index)]

In [39]:
subset_data.query("PAY_0 >= 1.50 & PAY_3 >= 1.00 & PAY_6 >= 1.00")[target].value_counts(normalize=False) 

default payment next month
1    787
0    237
Name: count, dtype: int64

,segment,count,events,rate
0,0,28976,5849.0,20.185671
1,5,1024,787.0,76.855469


In [26]:
def convert_interval_to_sql(rule_string):
    """
    Parses a string like 'PAY_0=[1.50, inf) & PAY_4=[0.50, inf)'
    and converts it to a SQL WHERE clause compatible string.
    """
    sql_conditions = []
    
    # Split by '&' to process each column's rule individually
    rules = rule_string.split('&')
    
    for rule in rules:
        rule = rule.strip()
        if not rule:
            continue
            
        # Separate the column name from the interval
        col_name, interval = rule.split('=')
        col_name = col_name.strip()
        interval = interval.strip()
        
        # Extract the brackets and the numeric values
        left_bracket = interval[0]
        right_bracket = interval[-1]
        
        # Remove brackets and split by comma to get lower and upper bounds
        values_str = interval[1:-1]
        lower_str, upper_str = [v.strip() for v in values_str.split(',')]
        
        col_conditions = []
        
        # Determine the lower bound condition (if it's not -inf)
        if lower_str.lower() != '-inf':
            # '[' means inclusive (>=), '(' means exclusive (>)
            operator = '>=' if left_bracket == '[' else '>'
            col_conditions.append(f"{col_name} {operator} {lower_str}")
            
        # Determine the upper bound condition (if it's not inf)
        if upper_str.lower() != 'inf':
            # ']' means inclusive (<=), ')' means exclusive (<)
            operator = '<=' if right_bracket == ']' else '<'
            col_conditions.append(f"{col_name} {operator} {upper_str}")
            
        # Combine conditions for this specific column using SQL 'AND'
        if col_conditions:
            sql_conditions.append(" AND ".join(col_conditions))
            
    # Combine all column rules into the final SQL string using 'AND'
    # Parentheses are added around variables that have both an upper and lower bound
    return " AND ".join(f"({cond})" if "AND" in cond else cond for cond in sql_conditions)

# --- Test the converter ---
input_string = first_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PAY_0=[1.50, inf) & PAY_4=[0.50, inf) & PAY_6=[1.00, inf)
Output SQL WHERE clause: PAY_0 >= 1.50 AND PAY_4 >= 0.50 AND PAY_6 >= 1.00


In [ ]:
# --- Test the converter ---
input_string = second_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PAY_0=[1.50, inf) & BILL_AMT4=[5756.00, inf) & SEX=[1.50, inf)
Output SQL WHERE clause: PAY_0 >= 1.50 AND BILL_AMT4 >= 5756.00 AND SEX >= 1.50


In [28]:
# --- Test the converter ---
input_string = third_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PAY_0=[0.50, inf) & PAY_2=[1.50, inf) & PAY_3=[1.50, inf)
Output SQL WHERE clause: PAY_0 >= 0.50 AND PAY_2 >= 1.50 AND PAY_3 >= 1.50


In [29]:
# --- Test the converter ---
input_string = forth_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PAY_0=[0.50, inf) & PAY_3=(-inf, 0.50) & SEX=(-inf, 1.50)
Output SQL WHERE clause: PAY_0 >= 0.50 AND PAY_3 < 0.50 AND SEX < 1.50


In [30]:
# --- Test the converter ---
input_string = fifth_rule
generated_sql = convert_interval_to_sql(input_string)

print("\n--- Auto-Generated SQL Filter ---")
print(f"Input:  {input_string}")
print(f"Output SQL WHERE clause: {generated_sql}")


--- Auto-Generated SQL Filter ---
Input:  PAY_2=[-0.50, inf) & PAY_AMT1=(-inf, 21.50)
Output SQL WHERE clause: PAY_2 >= -0.50 AND PAY_AMT1 < 21.50


In [31]:
import duckdb
result = duckdb.sql("""

SELECT CASE WHEN PAY_0 >= 1.50 AND PAY_4 >= 0.50 AND PAY_6 >= 1.00 THEN 1
WHEN PAY_0 >= 1.50 AND BILL_AMT4 >= 5756.00 AND SEX >= 1.50 THEN 2
WHEN PAY_0 >= 0.50 AND PAY_2 >= 1.50 AND PAY_3 >= 1.50 THEN 3
WHEN PAY_0 >= 0.50 AND PAY_3 < 0.50 AND SEX < 1.50 THEN 4
WHEN PAY_2 >= -0.50 AND PAY_AMT1 < 21.50 THEN 5
ELSE 0 END AS segment, 
COUNT(*) AS count,
SUM("default payment next month") AS events,
(SUM("default payment next month") / COUNT(*)) * 100 AS rate,
FROM data
GROUP BY segment

""").df()

result



,segment,count,events,rate
0,0,23936,3390.0,14.162767
1,1,1002,774.0,77.245509
2,2,1040,720.0,69.230769
3,3,1382,732.0,52.966715
4,4,1533,657.0,42.857143
5,5,1107,363.0,32.791328
